# Crude oil, from arrival to a signal

This is the research side of the platform. It is deliberately the only place in
the repository where anyone looks at a number and forms an opinion about it.

Two rules hold throughout, and they are the whole point of the platform existing:

1. **Every number here comes through `MarketData`.** No file is opened, no SQL is
   written, and nothing below knows where the parquet lives or how the roll is
   stored. A test in `tests/test_platform.py` greps these cells for
   `read_parquet`, `SELECT `, `store.query(`, `duckdb.connect` and `open(` and
   fails if any appears. That is a tripwire, not a proof: it is a substring
   check, and it cannot stop a researcher reaching the store through a helper.
   It is worth having because a shortcut taken once in a notebook is a shortcut
   copied into every notebook after it. The imports below — `mdp.model`,
   `mdp.recipes`, `mdp.charts` — are library code sitting on the same serving
   API, not a way around it.
2. **Nothing here sees data from after the moment being evaluated.** The
   point-in-time guarantees are enforced below the API, not by discipline up
   here, because discipline is not a control.

Operational health is not in this notebook. Whether the file arrived, whether a
vendor is drifting, whether the poller is alive: that is Prometheus and Grafana,
which is where the on-call rotation already lives. Mixing the two into one page
is how you get a dashboard nobody owns.

In [ ]:
import sys
from pathlib import Path

# So the notebook runs from a plain `jupyter lab` with nothing installed and no
# PYTHONPATH set. A notebook that needs a wrapper script is a notebook that
# stops getting opened.
src = Path.cwd().parent / "src"
if src.exists() and str(src) not in sys.path:
    sys.path.insert(0, str(src))

%matplotlib inline

import warnings

import matplotlib.pyplot as plt
import pandas as pd

from mdp.charts import GRID, SERIES
from mdp.model import make_features
from mdp.recipes import Recipe, comparison_table, run_recipe
from mdp.serve import MarketData
from mdp.storage import get_store

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

store = get_store()

# `consumer` is not decoration. Every read is checked against the entitlement
# policy and written to an access log, so "who looked at the licensed feed and
# when" is answerable without anyone having thought to instrument it.
md_api = MarketData(store=store, consumer="research")
md_api.catalog()

## 1. What the roll does to a price series

A futures contract expires. A continuous series has to splice contracts
together, and the splice leaves a jump that is an artefact of switching
contracts rather than anything the market did.

If you skip the adjustment, that artefact is a return, and every model you fit
will learn it. This is the single most common way a commodities backtest lies to
you, so it is the first thing worth showing.

In [ ]:
raw = md_api.get_continuous("CRUDEOIL", method="volume", adjust="none")
adj = md_api.get_continuous("CRUDEOIL", method="volume", adjust="ratio")

rolls = adj.loc[adj["is_roll"], "trade_date"]
print(f"{len(adj)} rows, {len(rolls)} rolls")

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True,
                         gridspec_kw={"height_ratios": [2, 1]})

axes[0].plot(raw["trade_date"], raw["close"], color=SERIES[1], lw=1.4, label="spliced, unadjusted")
axes[0].plot(adj["trade_date"], adj["adj_close"], color=SERIES[0], lw=1.4, label="back-adjusted")
for day in rolls:
    axes[0].axvline(day, color="#c9c8c1", lw=0.8, zorder=0)
axes[0].set_title("CRUDEOIL continuous series, front month by volume")
axes[0].legend(frameon=False, loc="upper left")

artefact = (raw["close"].pct_change().values - adj["adj_close"].pct_change()) * 100
axes[1].axhline(0, color="#c9c8c1", lw=0.8)
axes[1].vlines(adj["trade_date"], 0, artefact, color=SERIES[1], lw=1.2)
axes[1].set_ylabel("percentage points")
axes[1].set_title("Return that is purely an artefact of switching contracts",
                  fontsize=10, loc="left")
fig.tight_layout()
plt.show()

print(f"artefact is non-zero on {int((artefact.abs() > 1e-9).sum())} days "
      f"out of {len(artefact)}, largest {artefact.abs().max():.2f} percentage points")


The lower panel is the difference between the two return series, so it is zero on
every day except a roll. Those spikes are not price moves. They are the splice,
and the largest of them printed above is comfortably the biggest "move" in the
sample, occurring on a day the market did nothing unusual.

A model fitted on the unadjusted series learns to predict contract expiry, and it
will look like it has found something.

Two details in that chart are worth naming because both were wrong once. The roll
count is what a monthly contract can have, not what daily volume suggests: the
naive "most volume today" rule flips back and forth several times a month,
because volume does, and the forward-only rule is what turns that into roughly
twelve rolls a year. And every row plotted is a day the front contract actually
printed — a day it did not is dropped rather than carried forward, because a
carried-forward close is a flat day the exchange never published, and that
invented return goes straight into volatility and into every feature below.

## 2. Point in time, and why `as_of` is not optional

`get_continuous(as_of=...)` returns the series **as it was known on that date**.
It does that by reading the stored front-month map back — `contract_reference`
carries `trade_date` alongside `known_from` and `known_to`, and a refresh closes
a superseded row rather than deleting it.

The distinction matters more than it sounds. The tempting implementation is to
truncate the prices and re-run the roll rule, which is not the same thing at all:
re-running uses *today's* settlements, so a correction that arrived last week
silently changes which contract was front on a day that is already inside a
published backtest, and nothing records that the answer moved.

Contract specifications get the same treatment. A lot size that changed in March
means a position sized with today's multiplier is wrong by a constant factor for
every month before it, and constant factors are the hardest errors to see in a
P&L. `notional` raises rather than guessing when no spec version covers the date
asked for.

In [ ]:
cut = adj["trade_date"].iloc[len(adj) // 2]
as_known_then = md_api.get_continuous("CRUDEOIL", as_of=cut)
print(f"as of {pd.Timestamp(cut).date()}: {len(as_known_then)} rows "
      f"(today: {len(adj)})")

# The spec history in config/contract_specs.yml carries an illustrative lot
# size change in 2020, so the point-in-time lookup has something to find.
# The price data here starts later, which is exactly the situation that makes
# this dangerous: nothing in the price series tells you the multiplier moved.
for day in ("2018-06-30", "2025-06-30"):
    spec = md_api.get_specs("CRUDEOIL", as_of=day)
    print(f"{day}: lot size {spec.get('lot_size')}, tick {spec.get('tick_size')}")

px = float(adj["adj_close"].iloc[-1])
print(f"one lot at {px:,.1f} is worth "
      f"{md_api.notional('CRUDEOIL', px, 1):,.0f} using the spec of the day")

## 3. The other shape: tick data and the as-of join

End-of-day data is a table you can join on a date. Tick data is not, and the
difference is the whole reason a market data platform is harder than a
warehouse.

A decision made at 10:31:17 needs the price **at or before** that instant. Never
after, or the model is reading the future. Never a bar boundary, because
decisions do not arrive on the minute. That is an as-of join, and it is the
operation the serving layer exists to make correct once rather than correct in
each researcher's notebook.

`research_frame` also attaches trailing realised volatility, computed only from
bars that **end** at or before the decision time. That is stricter than it looks,
and the obvious version is wrong: a bar is stamped with the start of the interval
it covers, so a decision at 10:50:17 admitting the bar stamped 10:50:00 is
admitting a close built from trades after the decision. On this repo's own demo
data that single bar once moved realised volatility by a factor of 52.

In [ ]:
bars = md_api.get_bars("BTCUSDT", freq="1m")
decisions = pd.to_datetime(bars["bucket"], utc=True).iloc[5:-5:7] + pd.Timedelta(seconds=17.5)

frame = md_api.research_frame("BTCUSDT", decisions.tolist(), vol_window_minutes=10)
frame.head(8)

In [ ]:
window = frame["trade_ts"].notna()
lo, hi = frame.loc[window, "query_ts"].min(), frame.loc[window, "query_ts"].max()
view = bars[(pd.to_datetime(bars["bucket"], utc=True) >= lo - pd.Timedelta(minutes=2))
            & (pd.to_datetime(bars["bucket"], utc=True) <= hi + pd.Timedelta(minutes=2))]

fig, ax = plt.subplots(figsize=(11, 4.2))
ax.step(pd.to_datetime(view["bucket"], utc=True), view["close"], where="post",
        color="#c9c8c1", lw=1.2, label="1m bars")
ax.scatter(frame["query_ts"], frame["price"], s=34, color=SERIES[0], zorder=3,
           label="as-of price at each decision time")
for _, row in frame.dropna(subset=["trade_ts"]).head(40).iterrows():
    ax.plot([row["trade_ts"], row["query_ts"]], [row["price"], row["price"]],
            color=SERIES[0], lw=0.8, alpha=0.5)
ax.set_title("Every dot sits at or before its decision time, never after")
ax.legend(frameon=False, loc="upper left")
fig.tight_layout()
plt.show()

stale = frame["staleness_seconds"].dropna()
print(f"staleness: median {stale.median():.1f}s, worst {stale.max():.1f}s")

Every dot sits on the step at or before its own x position, never on the one
after. That is the invariant, and it is enforced in SQL inside the store rather
than by anyone remembering to do it.

Staleness here is sub-second, so it is invisible at this scale, which is the
normal case and the reason it has to be measured rather than eyeballed. The
platform returns it as a column on every row and nulls the price outright once
it exceeds the bound, because a price that is four hours old is a different
input from one that is four seconds old and a model cannot tell them apart.


## 4. Features

Twelve features, each registered by name so a recipe can ask for a subset and the
run record can say exactly which ones were used. No feature reads forward: the
registry is the place that invariant is enforced, and the only forward-looking
line in `model.py` is the target.

In [ ]:
features = make_features(adj, price_col="adj_close")
print(f"{features.shape[0]} rows, {features.shape[1]} columns")
features.tail(5)

In [ ]:
from mdp.model import DEFAULT_FEATURES, FEATURES

print(f"registry: {len(FEATURES)} features -> {sorted(FEATURES)}")
print(f"this run: {len(DEFAULT_FEATURES)} selected -> {DEFAULT_FEATURES}")

wide = make_features(adj, price_col="adj_close", features=sorted(FEATURES))
corr = wide[sorted(FEATURES)].corr()

fig, ax = plt.subplots(figsize=(8.5, 7))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=45, ha="right", fontsize=9)
ax.set_yticks(range(len(corr)), corr.columns, fontsize=9)
ax.set_title("All twelve, correlated: where the collinearity is, hence the ridge")
fig.colorbar(im, ax=ax, shrink=0.8)
fig.tight_layout()
plt.show()


The momentum and moving-average-distance blocks are visibly collinear, which is
the honest argument for ridge over ordinary least squares here. It is not a
default chosen because it usually works.

## 5. Walk-forward, and what it actually says

Every scenario in the recipe runs against **the same input frame**, fingerprinted
by row count, date range and a hash of the price series. If the data moved
between runs, the comparison would be measuring the data rather than the change
being tested, and nobody would notice.

The scoring is walk-forward: fit on a window, predict the next block, roll
forward, never look back. A single train/test split on a time series flatters a
model badly enough to be misleading.

There is also a **purge gap** between each training window and its test block,
sized to the scenario's own horizon. A target at horizon *h* is a forward return,
so the label on the last *h* training rows is computed from prices inside the
test block. Without the gap, the five-day scenario leaked five times as far as
the one-day scenario — and it leaked in the flattering direction, so the ranking
below would have been partly measuring leakage and reporting it as a result.

In [ ]:
recipe = Recipe.by_name("crudeoil_daily")
results = run_recipe(recipe, store, consumer="research")
table = comparison_table(results)
table

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.4))
ranked = table.sort_values("skill_vs_baseline")
colours = [SERIES[2] if v > 0 else SERIES[1] for v in ranked["skill_vs_baseline"]]
ax.barh(ranked["scenario"], ranked["skill_vs_baseline"] * 100, color=colours)
ax.axvline(0, color=GRID, lw=1)
ax.set_xlabel("RMSE improvement over the naive baseline, %")
ax.set_title("Scenario comparison, same input frame for every row")
fig.tight_layout()
plt.show()

## 6. What the result actually says

Every scenario scores **below** the naive baseline. That is the honest output and
it is worth stating plainly rather than hunting for a cut of the data where it
looks better.

Read the table and the chart above as follows:

- The baseline is a random walk: tomorrow equals today. On a daily horizon that
  is a genuinely hard benchmark, and a ridge on price-derived features does not
  clear it here. Direction accuracy hovers around fifty percent, which is a coin
  flip.
- `more_features` is the worst of the six, and by a wide margin rather than a
  hair. Eleven features over fourteen folds is more parameters chasing less data
  — a longer feature set needs a longer warm-up, so it gets fewer folds to learn
  from as well — and walk-forward is doing its job by punishing it. A single
  train/test split would very likely have made this scenario look like the
  winner. That ordering is the stable result here; which of the remaining five
  comes top shifts from run to run, and reading a story into that ordering would
  be reading noise.
- The end-of-day series is **synthetic**, generated with realistic defects so the
  pipeline's failure handling can be exercised offline. Nothing here is evidence
  about crude oil, and the series is generated relative to the day the notebook
  runs, so the exact figures above are not the ones somebody else will see.

The useful claim is not about the model. It is that six variants ran against a
byte-identical input frame, each with its own run id and data fingerprint, and
produced a comparison that can be trusted enough to act on. A negative result you
believe is worth considerably more than a positive one you do not, and most of
the engineering in this repository exists to make the difference.

## 7. The door is real

The last thing worth demonstrating is that the entitlement check is not
advisory. A consumer without a licence for the tick feed cannot read it, the
refusal is recorded, and this is enforced in the serving layer rather than in
each caller.

The access log below is itself behind the same door. It belongs to a dataset
called `platform_metadata` in `config/entitlements.yml`, along with the
catalogue, the check history and the vendor scorecard — because a table that
records who read the licensed feed and when is precisely the sort of thing that
should not be readable by whoever asks for it. It used to have no check on it at
all.

In [ ]:
restricted = MarketData(store=store, consumer="intern")
try:
    restricted.get_trades("BTCUSDT")
    print("read allowed")
except PermissionError as exc:
    print(f"refused, as it should be: {exc}")

md_api.access_log(limit=8)